# Repasando FastAPI paso a paso

En este Notebook repasaremos progresivamente los conceptos principales de FastAPI.

Cada sección retomará un concepto, lo pondrá en contexto e incorporará el código necesario para probarlo.


## 1. Instalación de FastAPI

FastAPI es el framework que utilizaremos para crear la API.

También instalaremos sus dependencias estándar, entre ellas Uvicorn, que permite ejecutar la aplicación como un servidor.

%pip install "fastapi[standard]"

## 2. Creación de la aplicación

Primero importamos la clase `FastAPI`.

Después creamos una instancia y la guardamos en la variable `app`. Este objeto representará nuestra aplicación.

In [1]:
from fastapi import FastAPI

app = FastAPI()

### Comprobación

Utilizamos 'type()' para comprobar que clase de objeto se guardo en la vaiable 'app'. 

In [2]:
type(app)

fastapi.applications.FastAPI

## 3. Ruta de prueba: Hello World

Esta ruta responde cuando alguien solicita la direccion principal de la API: '/'.

In [3]:
@app.get('/')
async def read_root():
    return{"message": "Hello, World"}

### Nota: rutas repetidas

FastAPI evalúa las rutas en el orden en que fueron registradas. Cuando recibe GET/, encuentra la primera coincidencia (read_root) y la ejecuta. Como ya encontró una ruta válida, no sigue buscando otra. Esto permite que rutas especificas se declaren antes que rutas variables, por ejemplo, /user/me antes de /user/{user_id}.

In [4]:
# Aplicamos para la misma ruta otra funcion a ver que pasa.
""" @app.get('/')
async def read_root2():
    return{"message": "Hello, World2"} """

' @app.get(\'/\')\nasync def read_root2():\n    return{"message": "Hello, World2"} '

Si dos funciones usan la misma combinación de método y ruta, por ejemplo `GET /`, FastAPI ejecuta la primera que fue registrada.
___
![Resultado de Hello World](Images/03%20Hello%20world.png)
___
Sin embargo, la documentación automática (`/docs`) solo puede mostrar una definición para `GET /`, por lo que termina mostrando la última.

Esto genera una inconsistencia: la documentación puede describir una ruta distinta de la que realmente se ejecuta. Por eso cada combinación de método HTTP y ruta debe ser única.
___
![Resultado de docs](Images/03%20Hello%20world%20docs.png)

## 4. Una segunda ruta: `/saludo`
Una ruta identifica una direccion de la API. `@app.get("/saludo")` registra la funcion siguiente para responder solicitudes `GET` a esa direccion.
El nombre `read_greeting` es elegido por nosotros; FastAPI usa la combinacion `GET /saludo` para encontrarla.

In [5]:
@app.get("/saludo")
async def read_greeting():
    return{"message": "Hola, Ramiro"}

![Hola Ramiro](Images\04%20Hola%20Ramiro.png)
___
![Hola Ramiro docs](Images\04%20Hola%20Ramiro%20docs.png)


## 5. Parametros de ruta

`{task_id}` representa una parte variable de la URL. En `/tasks/5`, FastAPI recibe 5 y lo entrega a `task_id`.
La anotacion `: init` egige un numero entero; si se escribe `/tasks/hola`, FastAPI devuelve un Error de validacion.

In [6]:
@app.get("/tasks/{task_id}")
async def read_task(task_id: int):
    return{"task_id": task_id}

![Tasks](Images\05%20Parametros%20ruta.png)
___
![Tasks docs](Images\05%20Parametros%20ruta%20docs.png)

## 6. Parámetros de consulta

Los parámetros de consulta son opcionales y modifican una consulta, por ejemplo `/tasks?completed=true&limit=5`; `?` inicia esa parte de la URL.

`bool | None = None` permite `True`, `False` o ausencia de filtro; así `None` significa “traer todas”.

Usamos `{task_id}` cuando el dato identifica obligatoriamente un recurso, como `/tasks/5`; usamos `?` para filtros, orden, paginación o límites que no cambian cuál es el recurso principal.

In [7]:
@app.get("/tasks")
async def read_tasks(completed: bool | None = None, limit: int = 10):
    return{"completed": completed, "limit": limit}

![Parametros consulta](Images\06%20Parametros%20consulta.png)
___
![Parametros consulta docs](Images\06%20Parametros%20consulta%20docs.png)

## 7. Validacion de parametros de consulta

`Query()` agrega reglas especificas para un parametro recibido desde la URL. `ge=1` significa "mayor o igual que 1" y `le=100`, "menor o igual que 100".

`Annotated` une el tipo `ìnt` con esas reglas; si `limit` queda fuera de ese rango. FastAPI rechaza la solicitud automaticamente con un error de validacion.

In [8]:
from typing import Annotated
from fastapi import Query

@app.get("/tasks-filtered")
async def read_filtered_tasks(completed:bool | None = None, limit: Annotated[int, Query(ge=1, le=100)]=10):
    return{"completed": completed, "limit": limit}

### Alcance de `Annotated` y `Query`
`Annotated` permite asociar información adicional a un tipo: aquí indica que `limit` es un `int` y que sus reglas provienen de `Query`.
Además de `ge` y `le`, `Query` puede validar longitudes, patrones, alias, valores obligatorios y descripciones para `/docs`.
El mismo mecanismo se usa más adelante con `Path`, `Header`, `Body` y `Depends`; no los aplicamos aún para incorporar una idea por vez.

![Validacion parametros consulta](Images\07%20Validacion%20parametros%20consulta.png)
___
![Validacion parametros consulta docs](Images\07%20Validacion%20parametros%20consulta%20docs.png)

## 8. Crear datos con `POST` y un modelo.

`POST` se usa para enviar datos nuevos a la API. `BaseModel` define la estructura que esperamos recibir en el cuerpo JSON  y FastAPI la valida automaticamente.

In [9]:
from pydantic import BaseModel

class TaskCreate(BaseModel):
    title: str
    completed: bool = False

@app.post("/tasks-created")
async def create_task(task: TaskCreate):
    return{"message": "Tarea creada", "task": task}

### ¿Para qué se usa `POST`?
`POST` se utiliza para crear recursos enviando datos al servidor: por ejemplo, una app web, móvil o frontend envía una tarea nueva como JSON.  
Al abrir `/tasks-created` en el explorador, este realiza una solicitud `GET`; como la ruta solo acepta `POST`, FastAPI responde `{"detail":"Method Not Allowed"}`.  
Para probarla usamos `/docs` → `POST /tasks-created` → **Try it out**, o herramientas como Postman, `curl` o código Python.
___
![Datos con POST](Images\08%20Post%20docs.png)

## 9. Validar el cuerpo de una solicitud.
`Field()` agrega reglas a cada dato del modelo. Asi evitamos crear tareas sin titulo o con un titulo demaciado corto.

Si el JSON no cumple estas reglas, FastAPI responde con un error `422` antes de ejecutarla funcion.

In [10]:
from pydantic import BaseModel, Field

class ValidatedTaskCreated(BaseModel):
    title: str = Field(min_length=3, max_length=100)
    completed: bool=False

@app.post("/tasks-validated")
async def create_validated_task(task: ValidatedTaskCreated):
    return{"message": "Tarea validada", "task": task}


Insertamos:

```json
{
  "title": "A",
  "completed": false
}
```

Sabemos que `"A"` tiene menos de tres caracteres.
___
![Datos con POST](Images\09%20Validar%20post%20docs.png)

## 10. Codigos de estado HTTP

Ademas de JSON, una API comunica el resultado mediante un codigo HTTP. `201 Created` indica que el servidor creó un recurso nuevo.

Usamos la constante `status.HTTP_201_CREATED` en lugar del numero `201` para que el codigo sea mas legible.

`200 OK` significa: “la operación salió bien”. Es el valor predeterminado de FastAPI si no indicamos otro.
`201 Created` significa algo más preciso: “la operación salió bien y además se creó un recurso nuevo”.

In [11]:
from fastapi import status

@app.post("/tasks-created-with-status", status_code=status.HTTP_201_CREATED)
async def create_task_with_status(task: ValidatedTaskCreated):
    return {"message": "Tarea creada correctamente", "task": task}

___
![Codigo HTTP](Images\10%20Codigo%20HTTP%20docs.png)

## 11. Modelo de respuesta

`response_model` define la estructura que la API enviara al cliente. Es util para documentar, validar y evitar devolver datos internos por error.

El modelo de entrada describe lo que recibimos; el de respuesta describe lo que entregamos. 

In [12]:
class TaskResponse(BaseModel):
    id: int
    title: str
    completed: bool

@app.post("/tasks-response-model", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
async def create_task_response(task: ValidatedTaskCreated):
    return{"id": 1, **task.model_dump()}

`task` es un objeto creado a partir de `ValidatedTaskCreate`. `task.model_dump()` lo convierte en un diccionario de Python con sus datos.
El operador `**` desempaqueta ese diccionario dentro de otro; por eso `{"id": 1, **task.model_dump()}` combina el `id` generado por el servidor con `title` y `completed` enviados por el cliente.
___
![Modelo respuesta](Images\11%20Modelo%20respuesta%20docs.png)

## 12. Almacenamiento temporal en memoria
Por ahora guardaremos las tareas en una lista de Python; funciona como una base de datos muy simple mientras el servidor está activo.

Al reiniciar el kernel, la lista se vacía: más adelante la reemplazaremos por una base de datos real. `next_task_id` genera identificadores consecutivos.

In [13]:
tasks_memory: list[TaskResponse]= []
next_task_id = 1

@app.post("/tasks-memory", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
async def create_task_in_memory(task: ValidatedTaskCreated):
    global next_task_id
    saved_task = TaskResponse(id=next_task_id, **task.model_dump())
    tasks_memory.append(saved_task)
    next_task_id += 1
    return saved_task

![Almacenamiento temporal comandos](Images\12%20Almacenamiento%20temporal%20command.png)
___
![Almacenamiento temporal docs](Images\12%20Almacenamiento%20temporal%20docs.png)

## 13. Listar tareas
La misma URL puede aceptar distintos metodos: `POST /tasks-memory` crea una tarea y `GET /tasks-memory` devuelve las existentes.

`response_model = list[TaskResponse]` indica que la respuesta será una lista; si todavia no creamos tareas, devuelve `[]`.

In [14]:
@app.get("/tasks-memory", response_model=list[TaskResponse])
async def read_tasks_memory():
    return tasks_memory

![Listar tareas](Images\13%20Listar%20tareas.png)
___
![Listar tareas docs](Images\13%20Listar%20tareas%20docs.png)

## 14. Buscar una tarea por su identificador
`{task_id}` recibe el numero escrito en la URL, por ejemplo `/tasks-memory/2`. Recorremos la lista hasta encontrar una tarea con ese `id`.

Por ahora, si no existe, devolvemos `null`; en el siguiente paso lo reemplazamos por el error HTTP correcto.

In [15]:
@app.get("/tasks-memory/{task_id}", response_model=TaskResponse | None)
async def read_task_memory(task_id: int):
    for task in tasks_memory:
        if task.id == task_id:
            return task
    return None

![Buscar tareas](Images\14%20Buscar%20tareas%20id%203.png)
___
![Buscar tareas](Images\14%20Buscar%20tareas%20id%207.png)
___
![Buscar tareas docs](Images\14%20Buscar%20tareas%20id%205%20docs.png)

## 15. Informar recursos inexistentes con `404`
`null` con codigo `200` puede confundir: la solicitud funcionó, pero no encontramos la tarea. `404 Not Found` expresa correctamente esta situacion.

`HTTPExeption` detiene la funcion y envia una respuesta de error con un mensaje para el cliente.

In [16]:
from fastapi import HTTPException

@app.get("/tasks-memory-safe/{task_id}", response_model=TaskResponse)
async def read_task_memory_safe(task_id:int):
    for task in tasks_memory:
        if task.id == task_id:
            return task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

![Recursos inexistentes](Images\15%20Recursos%20inexistentes.png)
___
![Recursos inexistentes docs](Images\15%20Recursos%20inexistentes%20docs.png)

## 16. Actualizar una tarea con `PUT`
`PUT` actualiza por completo un recurso existente identificado por su URL. A diferencia de `POST`, no crea un ID nuevo: reemplaza la tarea indicada.

El cliente debe enviar todos los datos necesarios; si no existe la tarea, devolvemos `404`.

In [17]:
@app.put("/tasks-memory-update/{task_id}", response_model=TaskResponse)
async def replace_task(task_id: int, task: ValidatedTaskCreated):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            updated_task = TaskResponse(id=task_id, **task.model_dump())
            tasks_memory[index] = updated_task
            return updated_task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

Para este caso primero vamos a hacer la actualizacion del `id=1` agregando `title:tarea modificada` y `completed:true`
___
![Actualizar tarea PUT](Images\16%20Actualizar%20tarea%20PUT.png)
___
Luego vamos a revisar en `/task-memory-safe/1` si fue modificado
___
![Actualizar tarea resultado](Images\16%20Actualizar%20tarea%20resultado.png)

## 17. Modificar solo algunos campos con `PATCH`
`PATCH` actualiza parcialmente un recurso: podemos cambiar solo `completed` sin reenviar el titulo. Por eso los campos de `TaskUpdate` son opcionales.
`exclude_unset = True` conserva únicamente los campos enviados por el cliente; evita que los campos omitidos reemplacen datos existentes por `None`

In [18]:
class TaskUpdate(BaseModel):
    title: str | None = Field(default = None, min_length = 3, max_length=100)
    completed: bool | None = None

@app.patch("/tasks-memory-partial/{task_id}", response_model=TaskResponse)
async def update_task_partially(task_id: int, task:TaskUpdate):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            updated_task = saved_task.model_copy(update=task.model_dump(exclude_unset=True))
            tasks_memory[index] = updated_task
            return updated_task
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Task not found")

### ¿Cuándo usar `PUT` y cuándo usar `PATCH`?
Usamos `PUT` cuando el cliente envía la versión completa que debe tener el recurso, reemplazando sus datos actuales.
Usamos `PATCH` cuando solo necesita cambiar uno o algunos campos, por ejemplo marcar una tarea como completada.
En APIs reales, `PATCH` suele ser más práctico para formularios de edición parcial o interruptores; `PUT` para reemplazos explícitos y completos.
___
![Actualizar tarea parcial docs](Images\17%20Actualizar%20tarea%20parcial%20docs.png)
___
![Actualizar tarea parcial](Images\17%20Actualizar%20tarea%20parcial.png)

## 18. Eliminar una tarea con `DELETE`
`DELETE` elimina el recurso identificado por la URL. Si se completa, respondemos `204 No Content`: la operacion fue correcta y no necesitamos devolver un cuerpo JSON.

Usamos `pop(index)` para quitar de la lista la tarea encontrada. Si el ID no existe, mantenemos el error `404`.

In [19]:
@app.delete("/tasks-memory-delete/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_task(task_id: int):
    for index, saved_task in enumerate(tasks_memory):
        if saved_task.id == task_id:
            tasks_memory.pop(index)
            return
    raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="task not found")

![Eliminar tarea antes](Images\18%20Eliminar%20tarea%20antes.png)
___
![Eliminar tarea docs](Images\18%20Eliminar%20tarea%20docs.png)
___
![Eliminar tarea despues](Images\18%20Eliminar%20tarea%20despues%201.png)
___
![Eliminar tarea safe despues](Images\18%20Eliminar%20tarea%20despues%202.png)

## 19. Reutilizar logica con dependencias
Evitar repetir codigo en varios endpoints usando el sistema de dependencias de FastAPI con `Depends()`

`Depends()` ejecuta una funcion antes de la ruta y entrega su resultado. Sirve para centralizar verificaciones compartidas, como permisos, usuarios autenticados o conecciones a una base de datos.

Aqui validamos una clave enviada como parámetro de consulta. Es una demostracion. En un proyecto real no guardariamos una clave sensible asi.

In [20]:
from fastapi import Depends

def verify_api_key(api_key:str):
    if api_key != "learning-key":
        raise HTTPException(status_code=status.HTTP_403_FORBIDDEN, detail="invalid API key")
    return api_key

@app.get("/tasks-protected", response_model=list[TaskResponse])
async def read_protected_tasks(api_key: str = Depends(verify_api_key)):
    return tasks_memory

___
![Reutilizar logica dependencias](Images\19%20Reutilizar%20logica%20dependencias.png)

In [21]:
import threading, uvicorn
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000), daemon=True).start()

## Para pruebas agregamos la siguiente libreria y URL base.

In [22]:
import httpx

base_url = "http://127.0.0.1:8000"

INFO:     Started server process [8372]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


### Caso 8 - Crear una tarea simple### Tambien puede hacerse por codigo.

In [23]:
response = httpx.post(
    f"{base_url}/tasks-created",
    json={"title": "Estudiar POST", "completed": False},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:51715 - "POST /tasks-created HTTP/1.1" 200 OK
200 {'message': 'Tarea creada', 'task': {'title': 'Estudiar POST', 'completed': False}}


### Caso 9 - Probar validación

In [24]:
valid_response = httpx.post(f"{base_url}/tasks-validated", json={"title": "Tarea válida"})
invalid_response = httpx.post(f"{base_url}/tasks-validated", json={"title": "A"})

print(valid_response.status_code, valid_response.json())
print(invalid_response.status_code, invalid_response.json())

INFO:     127.0.0.1:51716 - "POST /tasks-validated HTTP/1.1" 200 OK
INFO:     127.0.0.1:51717 - "POST /tasks-validated HTTP/1.1" 422 Unprocessable Entity
200 {'message': 'Tarea validada', 'task': {'title': 'Tarea válida', 'completed': False}}
422 {'detail': [{'type': 'string_too_short', 'loc': ['body', 'title'], 'msg': 'String should have at least 3 characters', 'input': 'A', 'ctx': {'min_length': 3}}]}


Caso 10 - Comprobar 201 Created

In [25]:
response = httpx.post(
    f"{base_url}/tasks-created-with-status",
    json={"title": "Comprobar código 201"},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:51719 - "POST /tasks-created-with-status HTTP/1.1" 201 Created
201 {'message': 'Tarea creada correctamente', 'task': {'title': 'Comprobar código 201', 'completed': False}}


### Caso 11 - Comprobar el modelo de respuesta

In [26]:
response = httpx.post(
    f"{base_url}/tasks-response-model",
    json={"title": "Comprobar respuesta estructurada", "completed": True},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:51721 - "POST /tasks-response-model HTTP/1.1" 201 Created
201 {'id': 1, 'title': 'Comprobar respuesta estructurada', 'completed': True}


### Caso 12 - Cargar seis tareas en memoria

In [27]:
titles = ["Leer documentación", "Crear rutas", "Probar POST", "Usar PUT", "Usar PATCH", "Documentar API"]

for title in titles:
    response = httpx.post(f"{base_url}/tasks-memory", json={"title": title})
    print(response.status_code, response.json())

INFO:     127.0.0.1:51722 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 1, 'title': 'Leer documentación', 'completed': False}
INFO:     127.0.0.1:51724 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 2, 'title': 'Crear rutas', 'completed': False}
INFO:     127.0.0.1:51725 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 3, 'title': 'Probar POST', 'completed': False}
INFO:     127.0.0.1:51727 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 4, 'title': 'Usar PUT', 'completed': False}
INFO:     127.0.0.1:51728 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 5, 'title': 'Usar PATCH', 'completed': False}
INFO:     127.0.0.1:51729 - "POST /tasks-memory HTTP/1.1" 201 Created
201 {'id': 6, 'title': 'Documentar API', 'completed': False}


### Caso 13 - Listar las tareas creadas

In [28]:
response = httpx.get(f"{base_url}/tasks-memory")
print(response.status_code)
print(response.json())

INFO:     127.0.0.1:51730 - "GET /tasks-memory HTTP/1.1" 200 OK
200
[{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}, {'id': 6, 'title': 'Documentar API', 'completed': False}]


### Caso 14 - Buscar una tarea por ID

In [29]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[0]["id"]

response = httpx.get(f"{base_url}/tasks-memory/{task_id}")
print(response.status_code, response.json())

INFO:     127.0.0.1:51732 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:51734 - "GET /tasks-memory/1 HTTP/1.1" 200 OK
200 {'id': 1, 'title': 'Leer documentación', 'completed': False}


### Caso 15 - Solicitar una tarea inexistente

In [30]:
response = httpx.get(f"{base_url}/tasks-memory-safe/-1")
print(response.status_code, response.json())

INFO:     127.0.0.1:51736 - "GET /tasks-memory-safe/-1 HTTP/1.1" 404 Not Found
404 {'detail': 'Task not found'}


### Caso 17 - Modificar parcialmente una tarea con PATCH

In [31]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[5]["id"]

response = httpx.patch(
    f"{base_url}/tasks-memory-partial/{task_id}",
    json={"completed": True},
)
print(response.status_code, response.json())

INFO:     127.0.0.1:51737 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:51739 - "PATCH /tasks-memory-partial/6 HTTP/1.1" 200 OK
200 {'id': 6, 'title': 'Documentar API', 'completed': True}


### Caso 18 - Eliminar una tarea con `DELETE`

In [32]:
tasks = httpx.get(f"{base_url}/tasks-memory").json()
task_id = tasks[-1]["id"]

response = httpx.delete(f"{base_url}/tasks-memory-delete/{task_id}")
print(response.status_code)

print(httpx.get(f"{base_url}/tasks-memory").json())

INFO:     127.0.0.1:51741 - "GET /tasks-memory HTTP/1.1" 200 OK
INFO:     127.0.0.1:51742 - "DELETE /tasks-memory-delete/6 HTTP/1.1" 204 No Content
204
INFO:     127.0.0.1:51743 - "GET /tasks-memory HTTP/1.1" 200 OK
[{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}]


### Caso 19 - Reutilizar lógica con dependencias.

In [33]:
response = httpx.get(f"{base_url}/tasks-protected", params = {"api_key": "learning-key"})
print(response.status_code, response.json())

response = httpx.get(f"{base_url}/tasks-protected", params = {"api_key": "wrong-key"})
print(response.status_code, response.json())

INFO:     127.0.0.1:51744 - "GET /tasks-protected?api_key=learning-key HTTP/1.1" 200 OK
200 [{'id': 1, 'title': 'Leer documentación', 'completed': False}, {'id': 2, 'title': 'Crear rutas', 'completed': False}, {'id': 3, 'title': 'Probar POST', 'completed': False}, {'id': 4, 'title': 'Usar PUT', 'completed': False}, {'id': 5, 'title': 'Usar PATCH', 'completed': False}]
INFO:     127.0.0.1:51745 - "GET /tasks-protected?api_key=wrong-key HTTP/1.1" 403 Forbidden
403 {'detail': 'invalid API key'}
